In [1]:
# ========== 1. Install dependencies ==========
!pip install -q unsloth wandb datasets huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 839.6 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 22.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 24.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 56.3 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 90.2 MB/s eta 0:00:00:0

In [ ]:
import os
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import Trainer
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import wandb

# ========== 1. SECURITY: Load credentials from environment ==========
HF_USERNAME = "TeslaInch"
HF_DATASET = "SCD-Instruction-Tuning"
HF_TOKEN = "your_api_key_here"
WANDB_API_KEY = "your_api_key_here"
WANDB_PROJECT = "scd-phi35-finetuning"

# Validate credentials exist
if not HF_TOKEN:
    raise ValueError("❌ HF_TOKEN not found. Set it: export HF_TOKEN='your_token'")
if not WANDB_API_KEY:
    raise ValueError("❌ WANDB_API_KEY not found. Set it: export WANDB_API_KEY='your_key'")

# ========== 2. Check CUDA availability ==========
if not torch.cuda.is_available():
    raise RuntimeError("❌ CUDA not available. GPU required for training.")
print(f"✅ Using GPU: {torch.cuda.get_device_name(0)}")

# ========== 3. Authenticate ==========
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
wandb.login(key=WANDB_API_KEY)

# ========== 4. Load dataset from Hugging Face ==========
try:
    # Removed split="train" to load the complete DatasetDict containing both train and validation
    dataset = load_dataset(
        f"{HF_USERNAME}/{HF_DATASET}",
        token=HF_TOKEN,
        trust_remote_code=False
    )
    print(f"✅ Loaded splits: {list(dataset.keys())}")
except Exception as e:
    raise RuntimeError(f"❌ Failed to load dataset: {e}")

# ========== 5. Validate dataset format ==========
required_fields = {"instruction", "input", "output"}
if not required_fields.issubset(set(dataset["train"].column_names)):
    raise ValueError(
        f"❌ Dataset missing required fields. Found: {dataset['train'].column_names}\n"
        f"Required: {required_fields}"
    )
print(f"✅ Dataset has correct schema: {dataset['train'].column_names}")

# ========== 6. Load model with Unsloth ==========
try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="microsoft/Phi-3.5-mini-instruct",
        max_seq_length=2048,
        load_in_4bit=True,
        device_map="auto",
    )
    print("✅ Model loaded successfully")
except Exception as e:
    raise RuntimeError(f"❌ Failed to load model: {e}")

# ========== 7. Apply LoRA adapters ==========
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=369,
)
print("✅ LoRA adapters applied")

# ========== 8. Format dataset using Phi-3 native template ==========
tokenizer = get_chat_template(
    tokenizer,
    chat_template="phi-3",
)

def format_chat_template(examples):
    """Convert dataset into standard conversational format"""
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        user_content = f"{inst}\n{inp}".strip() if inp else inst.strip()
        
        messages = [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": out.strip()}
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False))
    return {"text": texts}

try:
    # Map automatically runs on all splits in the DatasetDict
    dataset = dataset.map(format_chat_template, batched=True, remove_columns=dataset["train"].column_names)
    print(f"✅ Formatted train ({len(dataset['train'])} examples) and validation ({len(dataset['validation'])} examples) to Phi-3 format")
except Exception as e:
    raise RuntimeError(f"❌ Failed to format dataset: {e}")

# ========== 9. Setup training arguments ==========
training_args = SFTConfig(
    output_dir="./scd_phi35_adapter",
    dataset_text_field="text",          # Moved from trainer
    max_seq_length=2048,                # Moved from trainer
    num_train_epochs=6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    learning_rate=6e-5,
    lr_scheduler_type="cosine",     
    max_grad_norm=0.3,    
    weight_decay=0.01,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=50,
    eval_strategy="epoch",              # Run validation at the end of each epoch
    save_strategy="epoch",              # Save checkpoints at the end of each epoch
    save_total_limit=2,
    optim="adamw_8bit",
    seed=42,
    report_to=["wandb"],
)

# ========== 10. Create SFT Trainer ==========
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],        # Explicitly map train split
    eval_dataset=dataset["validation"],    # Explicitly map validation split
    args=training_args,                 # Now uses SFTConfig
    dataset_num_proc=2,
)

# ========== 11. Mask Prompts ==========
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|user|>\n",
    response_part="<|assistant|>\n",
)
print("✅ Input masking applied")

# ========== 12. Start training ==========
try:
    print("🚀 Starting training...")
    trainer.train()
    print("✅ Training completed successfully!")
except KeyboardInterrupt:
    print("⚠️  Training interrupted by user")
except Exception as e:
    print(f"❌ Training failed: {e}")
    raise

# ========== 13. Save model and tokenizer ==========
try:
    output_path = "./scd_phi35_adapter_final"
    model.save_pretrained(output_path)
    tokenizer.save_pretrained(output_path)
    print(f"✅ Model saved to {output_path}/")
except Exception as e:
    print(f"❌ Failed to save model: {e}")
    raise

# ========== 14. Cleanup ==========
wandb.finish()
print("✅ Training complete! Adapter ready for inference.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Using GPU: Tesla T4


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: tesla369 (tesla369-pdepth) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


README.md:   0%|          | 0.00/461 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/337k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/41.8k [00:00<?, ?B/s]

...

✅ Training completed successfully!


Unsloth: Restored added_tokens_decoder metadata in ./scd_phi35_adapter_final/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./scd_phi35_adapter_final.


✅ Model saved to ./scd_phi35_adapter_final/


eval/loss,█▁▁▇▁▄
eval/runtime,█▂▁▂▂▁
eval/samples_per_second,▁▇█▇▇█
eval/steps_per_second,▁▇█▇▇█
train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇█████
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇████
train/grad_norm,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▅▇███████▇▇▇▇▇▇▆▆▆▅▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁
train/loss,▂▁▁▂▁▄▄▇▇█▇▆▇▇█▆▆▆▇▇▆█▇█▆▇▇▇▆▆██▇▆▆▇▇▅▇▇
eval/loss,7.82111
eval/runtime,30.1226


✅ Training complete! Adapter ready for inference.
